# FinSight RAG Evaluation

Batch-evaluate the RAG pipeline against a set of test questions and visualise quality metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import requests

API = 'http://localhost:8000/api/v1'

# Customise these questions for your uploaded documents
TEST_QUESTIONS = [
    'What was the total revenue for the most recent fiscal year?',
    'What are the main risk factors disclosed?',
    'Are there any going concern risks mentioned?',
    'What is the debt-to-equity ratio?',
    'Were there any related party transactions?',
    'What dividends were declared?',
]

In [ ]:
rows = []
for q in TEST_QUESTIONS:
    r = requests.post(f'{API}/query', json={'question': q}, timeout=120)
    if r.status_code == 200:
        d = r.json()
        rows.append({
            'question':    q,
            'faithfulness':  d['faithfulness_score'],
            'relevancy':     d['answer_relevancy_score'],
            'fraud_score':   d['fraud_risk_score'],
            'flags':         len(d['fraud_flags']),
            'sources':       len(d['sources']),
            'latency_ms':    d['latency_ms'],
        })

df = pd.DataFrame(rows)
df

In [ ]:
print('=== Summary ===')
print(f'Mean faithfulness:  {df["faithfulness"].mean():.3f}')
print(f'Mean relevancy:     {df["relevancy"].mean():.3f}')
print(f'Mean latency:       {df["latency_ms"].mean():.0f} ms')
print(f'Queries with flags: {(df["flags"] > 0).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['faithfulness'].plot(kind='bar', ax=axes[0], title='Faithfulness', ylim=(0, 1))
df['relevancy'].plot(kind='bar',    ax=axes[1], title='Answer Relevancy', ylim=(0, 1))
df['latency_ms'].plot(kind='bar',   ax=axes[2], title='Latency (ms)')
for ax in axes:
    ax.set_xlabel('')
plt.tight_layout()
plt.show()